# AeroNetra — VisDrone Dataset Preparation

**Purpose:** Filter and remap a pre-converted YOLO-format VisDrone dataset to vehicle classes only.

> **Dataset used:** [`banuprasadb/visdrone-dataset`](https://www.kaggle.com/datasets/banuprasadb/visdrone-dataset)  
> This dataset is already in YOLO format (labels converted from VisDrone CSV by Ultralytics scripts).  
> Original VisDrone class IDs (0–11) are preserved in the labels, including pedestrians and ignored regions.  
> This notebook filters to vehicle classes only (IDs 3–10) and optionally remaps class IDs.

**Kaggle Setup:**

1. Add dataset: `banuprasadb/visdrone-dataset`**Outputs:** Filtered YOLO dataset saved to `/kaggle/working/visdrone_yolo/` — save as a Kaggle dataset to attach to training notebooks.

2. Accelerator: None (CPU is fine)
3. Internet: OFF (not needed)

In [ ]:
# ============================================================
# Cell 1: Configuration
# ============================================================
from pathlib import Path
import os

# Dataset root — matches banuprasadb/visdrone-dataset structure on Kaggle
# The dataset slug on Kaggle is 'visdrone-dataset', mounted at /kaggle/input/visdrone-dataset/
VISDRONE_ROOT = Path("/kaggle/input/visdrone-dataset/VisDrone_Dataset")

# Output directory (Kaggle working dir is saved as output)
OUTPUT_ROOT = Path("/kaggle/working/visdrone_yolo")

# Filter mode: "merged" (all vehicles → class 0) or "separate" (8 vehicle classes)
MODE = "separate"

# Splits to process
SPLITS = ["train", "val", "test"]

print(f"VisDrone root: {VISDRONE_ROOT}")
print(f"Output root:   {OUTPUT_ROOT}")
print(f"Mode:          {MODE}")
print(f"Root exists:   {VISDRONE_ROOT.exists()}")

In [ ]:
# ============================================================
# Cell 2: Discover the dataset directory structure
# ============================================================
# VisDrone datasets on Kaggle can have varying directory layouts.
# This cell auto-discovers the splits.

def discover_splits(root: Path) -> dict:
    """Discover VisDrone split directories (images + annotations)."""
    discovered = {}
    
    # Common patterns in VisDrone Kaggle uploads
    # NOTE: some uploads use 'annotations/', others use 'labels/' for annotation files
    patterns = [
        # Pattern 1: VisDrone2019-DET-{split}/images, .../annotations
        ("VisDrone2019-DET-{split}", "images", "annotations"),
        # Pattern 1b: VisDrone2019-DET-{split}/images, .../labels  (common Kaggle layout)
        ("VisDrone2019-DET-{split}", "images", "labels"),
        # Pattern 2: {split}/images, {split}/annotations
        ("{split}", "images", "annotations"),
        # Pattern 2b: {split}/images, {split}/labels
        ("{split}", "images", "labels"),
        # Pattern 3: Flat — images/{split}, annotations/{split}
        (None, "images/{split}", "annotations/{split}"),
        # Pattern 3b: Flat — images/{split}, labels/{split}
        (None, "images/{split}", "labels/{split}"),
    ]
    
    split_aliases = {
        "train": ["train", "VisDrone2019-DET-train"],
        "val": ["val", "VisDrone2019-DET-val"],
        "test": ["test", "test-dev", "VisDrone2019-DET-test-dev"],
    }
    
    for split, aliases in split_aliases.items():
        for alias in aliases:
            for parent_fmt, img_sub, ann_sub in patterns:
                if parent_fmt:
                    img_dir = root / parent_fmt.format(split=alias) / img_sub
                    ann_dir = root / parent_fmt.format(split=alias) / ann_sub
                else:
                    img_dir = root / img_sub.format(split=alias)
                    ann_dir = root / ann_sub.format(split=alias)
                
                if img_dir.exists() and ann_dir.exists():
                    discovered[split] = {"images": img_dir, "annotations": ann_dir}
                    break
            if split in discovered:
                break
    
    return discovered

# Discover
splits = discover_splits(VISDRONE_ROOT)

if not splits:
    # Fallback: list top-level contents for manual inspection
    print("Could not auto-discover splits. Directory contents:")
    for p in sorted(VISDRONE_ROOT.rglob("*")):
    print(f"{split:6s}: {n_img:5d} images, {n_ann:5d} annotation files")
            print(f"  [DIR]  {p.relative_to(VISDRONE_ROOT)}")
    raise FileNotFoundError("Update VISDRONE_ROOT or split patterns above.")

    print(f"         annotations: {paths['annotations']}")

for split, paths in splits.items():    print(f"         images:      {paths['images']}")

    n_img = len(list(paths['images'].glob('*.jpg')))    print(f"{split:6s}: {n_img:5d} images, {n_ann:5d} annotations")
    n_ann = len(list(paths['annotations'].glob('*.txt')))

In [ ]:
# ============================================================
# Cell 3: YOLO label class-filter (vehicle classes only)
# ============================================================
# banuprasadb/visdrone-dataset is already in YOLO format.
# Labels use the original VisDrone class IDs (0-11):
#   0=ignored, 1=pedestrian, 2=people, 3=bicycle, 4=car, 5=van,
#   6=truck, 7=tricycle, 8=awning-tricycle, 9=bus, 10=motor, 11=others
#
# This cell filters to vehicle classes (3-10) and remaps IDs for training.

import shutil
from typing import Optional

# Vehicle class IDs in the original VisDrone scheme
VISDRONE_VEHICLE_IDS = {3, 4, 5, 6, 7, 8, 9, 10}

# Separate mode: remap to 8 classes starting from 0
_SEPARATE_MAP = {4: 0, 5: 1, 6: 2, 7: 3, 8: 4, 9: 5, 10: 6, 3: 7}
SEPARATE_CLASS_NAMES = {
    0: "car", 1: "van", 2: "truck", 3: "tricycle",
    4: "awning-tricycle", 5: "bus", 6: "motor", 7: "bicycle",
}
MERGED_CLASS_NAMES = {0: "vehicle"}


def map_class(visdrone_cls_id: int, mode: str) -> Optional[int]:
    """Map a VisDrone class ID to an AeroNetra vehicle class ID.

    Returns None for non-vehicle classes (they are discarded).
    """
    if visdrone_cls_id not in VISDRONE_VEHICLE_IDS:
        return None
    return 0 if mode == "merged" else _SEPARATE_MAP[visdrone_cls_id]


def filter_split(
    images_dir: Path, labels_dir: Path, output_dir: Path, mode: str
) -> dict:
    """Filter YOLO labels to vehicle classes only, remap IDs, symlink images.

    Args:
        images_dir:  Source images directory.
        labels_dir:  Source YOLO labels directory (same-stem .txt files).
        output_dir:  Output root for this split (images/ and labels/ created inside).
        mode:        'merged' or 'separate'.

    Returns:
        Stats dict with counts of processed images and annotations.
    """
    out_img = output_dir / "images"
    out_lbl = output_dir / "labels"
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)

    stats = {
        "total_images": 0,
        "processed": 0,
        "valid_annotations": 0,
        "filtered_annotations": 0,
        "missing_labels": 0,
    }

    image_paths = sorted(images_dir.glob("*.jpg")) + sorted(images_dir.glob("*.png"))
    stats["total_images"] = len(image_paths)

    for img_path in image_paths:
        lbl_path = labels_dir / f"{img_path.stem}.txt"

        # Symlink / copy image regardless of whether a label file exists
        dst_img = out_img / img_path.name
        if not dst_img.exists():
            try:
                os.symlink(img_path.resolve(), dst_img)
            except OSError:
                shutil.copy2(img_path, dst_img)

        if not lbl_path.exists():
            stats["missing_labels"] += 1
            continue

        new_lines = []
        with open(lbl_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                orig_cls = int(parts[0])
                new_cls = map_class(orig_cls, mode)
                if new_cls is None:
                    stats["filtered_annotations"] += 1
                    continue
                # Keep only the 5 standard YOLO fields: class xc yc w h
                new_lines.append(f"{new_cls} {' '.join(parts[1:5])}\n")
                stats["valid_annotations"] += 1

        with open(out_lbl / f"{img_path.stem}.txt", "w", encoding="utf-8") as f:
            f.writelines(new_lines)

        stats["processed"] += 1

    return stats


print("Class-filter functions defined.")
print(f"Vehicle classes (VisDrone IDs): {sorted(VISDRONE_VEHICLE_IDS)}")


In [ ]:
# ============================================================
# Cell 4: Filter all splits
# ============================================================
import time

all_stats = {}

for split_name in SPLITS:
    if split_name not in splits:
        print(f"⚠️  Split '{split_name}' not found in dataset — skipping.")
        continue

    split_info = splits[split_name]
    out_dir = OUTPUT_ROOT / split_name

    print(f"\n{'='*60}")
    print(f"Filtering: {split_name}")
    print(f"{'='*60}")

    t0 = time.time()
    stats = filter_split(
        images_dir=split_info["images"],
        labels_dir=split_info["annotations"],
        output_dir=out_dir,
        mode=MODE,
    )
    elapsed = time.time() - t0

    all_stats[split_name] = stats
    print(f"  Total images:           {stats['total_images']}")
    print(f"  Processed:              {stats['processed']}")
    print(f"  Vehicle annotations:    {stats['valid_annotations']}")
    print(f"  Filtered (non-vehicle): {stats['filtered_annotations']}")
    print(f"  Missing labels:         {stats['missing_labels']}")
    print(f"  Time:                   {elapsed:.1f}s")


In [ ]:
# ============================================================
# Cell 5: Generate dataset.yaml for Ultralytics training
# ============================================================
import yaml

class_names = SEPARATE_CLASS_NAMES if MODE == "separate" else MERGED_CLASS_NAMES
nc = len(class_names)

# NOTE: 'path' is set to '.' (relative) so the yaml is portable.
# When Ultralytics loads it, it resolves paths relative to the yaml's location.
# This makes the dataset work whether it lives at /kaggle/working/visdrone_yolo
# (this notebook) or /kaggle/input/<slug>/visdrone_yolo (next notebook).
dataset_yaml = {
    "path": ".",
    "train": "train/images",
    "val": "val/images",
    "test": "test/images" if "test" in all_stats else "",
    "nc": nc,
    "names": class_names,
}

yaml_path = OUTPUT_ROOT / "dataset.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False, sort_keys=False)

print(f"Dataset YAML saved to: {yaml_path}")
print(f"Classes ({nc}): {class_names}")
print()
print(yaml_path.read_text())


In [ ]:
# ============================================================
# Cell 6: Verify output structure + sample labels
# ============================================================

print("Output directory structure:")
for split_name in SPLITS:
    split_dir = OUTPUT_ROOT / split_name
    if not split_dir.exists():
        continue
    n_img = len(list((split_dir / "images").glob("*")))
    n_lbl = len(list((split_dir / "labels").glob("*.txt")))
    print(f"  {split_name}/images: {n_img}  |  {split_name}/labels: {n_lbl}")

# Show a sample label
sample_label = next((OUTPUT_ROOT / "train" / "labels").glob("*.txt"), None)
if sample_label:
    print(f"\nSample label ({sample_label.name}):")
    lines = sample_label.read_text().strip().split("\n")
    for line in lines[:10]:
        parts = line.split()
        cls_id = int(parts[0])
        cls_name = class_names.get(cls_id, "?")
        print(f"  class={cls_id} ({cls_name})  xc={parts[1]}  yc={parts[2]}  w={parts[3]}  h={parts[4]}")
    if len(lines) > 10:
        print(f"  ... ({len(lines) - 10} more lines)")

In [ ]:
# ============================================================
# Cell 7: Class distribution visualization
# ============================================================
import matplotlib.pyplot as plt
from collections import Counter

# Count classes across the training set
class_counter = Counter()
train_labels_dir = OUTPUT_ROOT / "train" / "labels"

if train_labels_dir.exists():
    for label_file in train_labels_dir.glob("*.txt"):
        for line in label_file.read_text().strip().split("\n"):
            if line.strip():
                cls_id = int(line.split()[0])
                class_counter[cls_id] += 1

    names = [class_names.get(i, str(i)) for i in sorted(class_counter.keys())]
    counts = [class_counter[i] for i in sorted(class_counter.keys())]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(names, counts, color="steelblue", edgecolor="black")
    ax.set_xlabel("Vehicle Class")
    ax.set_ylabel("Number of Annotations")
    ax.set_title(f"VisDrone Training Set — Class Distribution ({MODE} mode)")
    ax.bar_label(bars, fmt="%d", fontsize=8)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig("/kaggle/working/class_distribution.png", dpi=150)
    plt.show()
    print(f"Total annotations: {sum(counts)}")
else:
    print("No training labels found.")

In [ ]:
# ============================================================
# Cell 8: Summary
# ============================================================
print("="*60)
print("DATASET PREPARATION COMPLETE")
print("="*60)
print(f"")
print(f"Output:       {OUTPUT_ROOT}")
print(f"YAML config:  {yaml_path}")
print(f"Mode:         {MODE} ({nc} classes)")
print(f"")
print("Next steps:")
print("  1. Save this notebook output as a Kaggle dataset")
print("     → New Dataset → name it 'aeronetra-visdrone-yolo'")
print("  2. Attach that dataset to the training notebook (02_model_training)")
print("  3. The training notebook expects the dataset at:")
print(f"     /kaggle/input/aeronetra-visdrone-yolo/visdrone_yolo/")